# Resultados del Exploiter

Lee los archivos que deja una corrida de entrenamiento terminada
(`accelerate launch ... train_reinforce_patched.py`, sin cambios en ese
flujo) y arma las mismas tablas que ya usamos para leer estas corridas a
mano durante la integración: resumen por proceso, categorías combinadas
ordenadas por `S(c)`, tasa de respuestas vacías/con forma de error (el
chequeo que originalmente destapó el problema de crédito de Anthropic), y
diversidad temática de las categorías generadas (el chequeo que destapó el
bug de orden de hooks).

No entrena nada ni carga ningún modelo -- solo lee JSON/JSONL ya escritos
en disco.

In [ ]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd

pd.set_option('display.max_colwidth', 90)

EXPERIMENTS_DIR = Path.cwd().parent / 'experiments'


def discover_runs(experiments_dir):
    """Group sibling run folders (4 parallel processes share one timestamp
    prefix, e.g. 20260721T194906Z-08338b/.../-b66eaf/) by that prefix."""
    groups = {}
    for d in sorted(experiments_dir.glob('2*Z-*')):
        if not d.is_dir() or not (d / 'config.json').exists():
            continue
        prefix = d.name.split('-')[0]
        groups.setdefault(prefix, []).append(d)
    return groups


RUNS = discover_runs(EXPERIMENTS_DIR)
print(f'{len(RUNS)} corridas encontradas en {EXPERIMENTS_DIR}:')
for prefix, folders in RUNS.items():
    print(f'  {prefix}: {len(folders)} proceso(s)')

# Cambiar aca si se quiere mirar una corrida distinta a la mas reciente
RUN_PREFIX = sorted(RUNS)[-1] if RUNS else None
FOLDERS = RUNS.get(RUN_PREFIX, [])
print(f'\nUsando: {RUN_PREFIX} ({len(FOLDERS)} proceso(s))')

## Resumen por proceso

In [ ]:
rows = []
for d in FOLDERS:
    cfg = json.loads((d / 'config.json').read_text(encoding='utf-8'))
    fr = json.loads((d / 'failure_report.json').read_text(encoding='utf-8')) if (d / 'failure_report.json').exists() else {}
    cats = fr.get('categories', [])
    rows.append({
        'proceso': d.name.split('-', 1)[1] if '-' in d.name else d.name,
        'juez': ('heuristico' if cfg.get('use_heuristic_judge') else (cfg.get('judge_model_name_or_path') or cfg.get('model_name_or_path'))),
        'categorias_por_step': cfg.get('categories_per_step'),
        'queries_por_categoria': cfg.get('queries_per_category'),
        'max_steps': cfg.get('max_steps'),
        'n_categorias': fr.get('n_categories', len(cats)),
        'n_pasaron_umbral': fr.get('n_passing', sum(1 for c in cats if c.get('passes_threshold'))),
        'max_S': max((c['mean_S'] for c in cats), default=0.0),
    })
pd.DataFrame(rows)

## Categorías combinadas (todos los procesos de esta corrida), ordenadas por S(c)

In [ ]:
all_cats = []
for d in FOLDERS:
    fr_path = d / 'failure_report.json'
    if not fr_path.exists():
        continue
    fr = json.loads(fr_path.read_text(encoding='utf-8'))
    for c in fr.get('categories', []):
        all_cats.append({
            'proceso': d.name.split('-', 1)[1] if '-' in d.name else d.name,
            'categoria': c['category'][:80],
            'mean_S': round(c['mean_S'], 4),
            'n_evals': c.get('n_evals'),
            'pasa_umbral': c.get('passes_threshold'),
        })

df_cats = pd.DataFrame(all_cats).sort_values('mean_S', ascending=False) if all_cats else pd.DataFrame()
df_cats

## Tasa de respuestas vacías / con forma de error

Este es el chequeo que originalmente destapó el problema de crédito de
Anthropic: una respuesta vacía o con forma de error API no es "el asistente
resistió", es "no hubo nada que evaluar". Un `S(c)` en 0.0000 solo es un
resultado real si esta tasa es baja.

In [ ]:
import sys
sys.path.insert(0, str(Path.cwd().parent / 'scripts'))
from _hf_router_judge import _looks_like_api_error  # misma definicion que usa el juez real

total = 0
malas = 0
por_proceso = []
for d in FOLDERS:
    rd_path = d / 'roast_dataset.jsonl'
    if not rd_path.exists():
        continue
    lines = rd_path.read_text(encoding='utf-8').splitlines()
    n_malas = sum(1 for l in lines if _looks_like_api_error(json.loads(l).get('response') or ''))
    total += len(lines)
    malas += n_malas
    por_proceso.append({
        'proceso': d.name.split('-', 1)[1] if '-' in d.name else d.name,
        'respuestas': len(lines),
        'vacias_o_error': n_malas,
        'tasa': f'{100 * n_malas / len(lines):.1f}%' if lines else 'n/a',
    })

print(f'TOTAL: {malas}/{total} ({100 * malas / max(total, 1):.1f}%) vacias o con forma de error\n')
pd.DataFrame(por_proceso)

## Diversidad temática de las categorías generadas

Cuántas categorías *distintas* (texto exacto) aparecen frente al total de
evaluaciones. Un número muy bajo (todo colapsando a 1-2 categorías
repetidas) es la señal que destapó el bug de orden de hooks -- el
generador no estaba explorando las debilidades de mayor peso del profile,
solo repetía la primera alfabéticamente.

In [ ]:
if all_cats:
    textos = [c['categoria'] for c in all_cats]
    n_unique = len(set(textos))
    print(f'categorias distintas: {n_unique}/{len(textos)} ({100 * n_unique / len(textos):.1f}% de diversidad)\n')
    print('mas frecuentes:')
    for texto, n in Counter(textos).most_common(10):
        print(f'  [{n}x] {texto}')
else:
    print('sin categorias para esta corrida')

## Notas

- **`n_evals` chico**: categorías con pocas evaluaciones son ruidosas — el
  asistente es estocástico (no contesta siempre igual a la misma
  pregunta), por eso `S(c)` penaliza la inconsistencia en vez de usar el
  promedio crudo.
- **Un `S(c)`/`mean_S` en 0.0000 no es automáticamente "el asistente no
  falla"** — revisar primero la tasa de vacías/error de la sección
  anterior antes de sacar esa conclusión.
- **Baja diversidad de categorías no es necesariamente un bug** — puede
  ser señal real de que el generador (entrenado o heurístico) converge a
  un tema, pero vale la pena mirar si ese tema es el de mayor debilidad
  real del profile o uno secundario.